In [1]:
import os, math, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models

In [ ]:
# --- Dataset paths ---
csv_path = "../osv-5m_subset/train_subset.csv"           # CSV path
img_dir  = "../osv-5m_subset/train"                      # folder with images

NUM_SAMPLES = 20
RANDOM_PICK = False     # set True to randomize; False = take first N
SEED = 42
IMAGE_SIZE = 224        # image size for model input
CHUNKSIZE = 200_000     # streaming chunk size for the CSV
maybe_filename_cols = ["image_path","filepath","path","relative_path","filename","image","file"]
VAL_SPLIT = 0.2
BATCH_SIZE = 8
LR = 1e-3
EPOCHS = 1

random.seed(SEED)

IMG_DIR = Path(img_dir)
assert IMG_DIR.exists(), f"Image dir not found: {IMG_DIR.resolve()}"

In [ ]:
# Pick images that exist on disk, pick images first, and then use the id to find metadata
EXts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
all_imgs = [p for p in IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXts]
assert len(all_imgs) > 0, f"No images found in {IMG_DIR}"

if RANDOM_PICK:
    random.shuffle(all_imgs)
picked_imgs = all_imgs[:NUM_SAMPLES]
picked_stems = {p.stem for p in picked_imgs}
picked_basenames = {p.name for p in picked_imgs}

print(f"Found {len(all_imgs)} images; using {len(picked_imgs)}")
print("Sample picks:", [p.name for p in picked_imgs[:5]])

# Stream the CSV and keep rows that match chosen images
kept_rows = []
for chunk in pd.read_csv(csv_path, chunksize=CHUNKSIZE):
    # Ensure lat/lon exist
    if not {"latitude","longitude"}.issubset(chunk.columns):
        raise RuntimeError("CSV must contain 'latitude' and 'longitude' columns.")

    mask_id = pd.Series(False, index=chunk.index)
    if "id" in chunk.columns:
        ids_as_str = chunk["id"].astype(str)
        mask_id = ids_as_str.isin(picked_stems)

    mask_fn = pd.Series(False, index=chunk.index)
    for col in maybe_filename_cols:
        if col in chunk.columns:
            col_vals = chunk[col].astype(str)
            m = col_vals.isin(picked_basenames)
            if not m.any():
                m = col_vals.map(lambda s: Path(s).name).isin(picked_basenames)
            mask_fn = mask_fn | m

    sub = chunk[mask_id | mask_fn].copy()
    if len(sub):
        keep_cols = [c for c in ["id","latitude","longitude"] + maybe_filename_cols if c in sub.columns]
        kept_rows.append(sub[keep_cols])

    # Stop early if we probably have enough candidates
    if sum(len(k) for k in kept_rows) >= NUM_SAMPLES * 2:
        break

if not kept_rows:
    raise RuntimeError("No metadata rows matched the chosen images. Check which CSV column stores filenames or verify you're using the correct CSV split.")

meta = pd.concat(kept_rows, ignore_index=True)

# uild df_small in the order of picked images
records = []
for p in picked_imgs:
    stem = p.stem
    row = None

    if "id" in meta.columns:
        hit = meta[meta["id"].astype(str) == stem]
        if len(hit):
            row = hit.iloc[0]

    if row is None:
        for col in [c for c in maybe_filename_cols if c in meta.columns]:
            hit = meta[meta[col].astype(str) == p.name]
            if len(hit):
                row = hit.iloc[0]; break
            hit = meta[meta[col].astype(str).map(lambda s: Path(s).name) == p.name]
            if len(hit):
                row = hit.iloc[0]; break

    if row is not None:
        try:
            lat = float(row["latitude"]); lon = float(row["longitude"])
            records.append({"image_path": str(p), "latitude": lat, "longitude": lon})
        except Exception:
            pass  # skip malformed rows

df_small = pd.DataFrame(records)
print(f"Matched {len(df_small)} / {len(picked_imgs)} picked images with metadata.")
display(df_small.head())

# Save for downstream steps (optional)
out_csv = Path("./df_small_images_first.csv")
df_small.to_csv(out_csv, index=False)
print("Saved:", out_csv.resolve())

Found 10000 images; using 20
Sample picks: ['788169555428235.jpg', '748002415887677.jpg', '975409006608878.jpg', '511970569954671.jpg', '1305258626637911.jpg']
Matched 20 / 20 picked images with metadata.


,image_path,latitude,longitude
0,../osv-5m_subset/train/788169555428235.jpg,38.997892,-95.319012
1,../osv-5m_subset/train/748002415887677.jpg,17.340472,99.438576
2,../osv-5m_subset/train/975409006608878.jpg,28.041546,69.782677
3,../osv-5m_subset/train/511970569954671.jpg,43.578574,-72.967603
4,../osv-5m_subset/train/1305258626637911.jpg,-34.657155,-58.747702


Saved: /work/cssema416/202610/28/GeoLocSFTTest/df_small_images_first.csv


In [12]:
# Prepare Dataset and Transforms
class GeoTiny(Dataset):
    def __init__(self, df_small, image_size=224):
        self.items = df_small.to_dict("records")
        self.tfm = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        rec = self.items[i]
        with Image.open(rec["image_path"]) as im:
            im = im.convert("RGB")
            x = self.tfm(im)
        y = torch.tensor([rec["latitude"], rec["longitude"]], dtype=torch.float32)
        return x, y, rec["image_path"]

dataset = GeoTiny(df_small, image_size=IMAGE_SIZE)
n_total = len(dataset)
n_val = max(1, int(n_total * VAL_SPLIT))
n_train = max(1, n_total - n_val)
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

len(train_ds), len(val_ds)

(16, 4)

In [13]:
# Model (ResNet18 frozen) and Haversine loss

EARTH_RADIUS_KM = 6371.0

class HaversineLoss(nn.Module):
    def forward(self, z_pred, target_deg):
        z_lat, z_lon = z_pred[:, 0], z_pred[:, 1]
        lat_pred = (math.pi/2) * torch.tanh(z_lat)
        lon_pred = (math.pi) * torch.tanh(z_lon)
        lat_t = torch.deg2rad(target_deg[:, 0])
        lon_t = torch.deg2rad(target_deg[:, 1])
        dlat = lat_pred - lat_t
        dlon = lon_pred - lon_t
        a = torch.sin(dlat/2)**2 + torch.cos(lat_t)*torch.cos(lat_pred)*torch.sin(dlon/2)**2
        c = 2 * torch.atan2(torch.sqrt(a + 1e-9), torch.sqrt(1 - a + 1e-9))
        return (EARTH_RADIUS_KM * c).mean()

def z_to_deg(z):
    z_lat, z_lon = z[:, 0], z[:, 1]
    lat = (90.0)  * torch.tanh(z_lat)
    lon = (180.0) * torch.tanh(z_lon)
    return torch.stack([lat, lon], dim=1)

class TinyGeoModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        body = list(backbone.children())[:-1]  # keep conv->GAP
        self.backbone = nn.Sequential(*body)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 2),
        )
    def forward(self, x):
        f = self.backbone(x)        # (B,512,1,1)
        f = torch.flatten(f, 1)     # (B,512)
        return self.head(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model   = TinyGeoModel().to(device)
loss_fn = HaversineLoss()
opt     = torch.optim.AdamW(model.head.parameters(), lr=LR)
model

TinyGeoModel(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats

In [14]:
# Do a simple train on it
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.train()
for xb, yb, _ in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(set_to_none=True)
    z = model(xb)
    loss = loss_fn(z, yb)
    loss.backward()
    opt.step()
float(loss)

/tmp/ipykernel_2545449/3064543754.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  float(loss)


8101.01025390625

In [16]:
def evaluate(model, loader, device):
    model.eval()
    preds, trues, paths = [], [], []
    with torch.no_grad():
        for xb, yb, pb in loader:
            xb = xb.to(device)
            z = model(xb)
            yhat = z_to_deg(z).cpu().numpy()
            preds.append(yhat)
            trues.append(yb.numpy())
            paths.extend(pb)
    if not preds:
        return {"count": 0}, pd.DataFrame()
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    def hav(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1; dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return 6371.0*c
    errs = np.array([hav(p[0], p[1], t[0], t[1]) for p, t in zip(preds, trues)])
    metrics = {
        "count": int(len(errs)),
        "mean_km": float(np.mean(errs)),
        "median_km": float(np.median(errs)),
        "p100_within": float(np.mean(errs <= 100.0)),
        "p750_within": float(np.mean(errs <= 750.0)),
    }
    df_pred = pd.DataFrame({
        "image_path": paths,
        "pred_lat": preds[:,0], "pred_lon": preds[:,1],
        "true_lat": trues[:,0], "true_lon": trues[:,1],
        "error_km": errs
    })
    return metrics, df_pred

metrics, preds = evaluate(model, val_loader, device)
display(metrics); display(preds.head())

out_dir = Path("./simple_run_images_first")
out_dir.mkdir(parents=True, exist_ok=True)

preds_path = out_dir / "preds.csv"
metrics_path = out_dir / "metrics.txt"
torch.save(model.state_dict(), out_dir / "model.pt")

preds.to_csv(preds_path, index=False)
with open(metrics_path, "w") as f:
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")

print("Saved:")
print(" -", preds_path)
print(" -", metrics_path)
print(" -", out_dir / "model.pt")

{'count': 4,
 'mean_km': 4937.9365234375,
 'median_km': 5171.5419921875,
 'p100_within': 0.0,
 'p750_within': 0.0}

,image_path,pred_lat,pred_lon,true_lat,true_lon,error_km
0,../osv-5m_subset/train/226862292235639.jpg,89.684921,-166.219513,41.270576,-73.210693,5420.388672
1,../osv-5m_subset/train/526446238768771.jpg,89.946785,-174.604935,55.823822,48.723961,3804.523682
2,../osv-5m_subset/train/1613737665683830.jpg,89.793777,-166.980850,39.792320,-8.750545,5604.139648
3,../osv-5m_subset/train/457614968877032.jpg,89.967499,-171.466415,45.708233,-121.511230,4922.695312


Saved:
 - simple_run_images_first/preds.csv
 - simple_run_images_first/metrics.txt
 - simple_run_images_first/model.pt
